In [10]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [11]:
# Node Function
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from langchain_groq import ChatGroq

model = ChatGroq( model="openai/gpt-oss-20b", api_key=os.environ.get("GROQ_API_KEY"))

In [12]:
# state

class LLMState(TypedDict):
    topic:str
    outline:str
    blog_post:str


In [13]:

# function

def generate_outline(state: LLMState) -> LLMState:
    topic = state["topic"]

    # prompt
    prompt = f"""
You are an expert SEO content strategist.
Create a detailed, well-structured blog post OUTLINE on the topic: "{topic}".

Requirements:
- Provide a compelling SEO-friendly title
- Write 2–3 introduction bullet points
- Include clear H2 and H3 headings
- Add key talking points under each heading
- Add examples or case-study suggestions where relevant
- End with a conclusion + strong takeaway
- Include 5–7 FAQs matching real user search intent
- Keep the outline logical, engaging, and optimized for readability

Return only the outline, nothing else.
"""

    state["outline"] = model.invoke(prompt).content
    return state


def generate_detailed_blog(state: LLMState) -> LLMState:
    outline = state["outline"]

    prompt = f"""
You are an expert blog writer and SEO specialist.
Using the following outline, write a fully detailed, engaging, SEO-optimized blog post:

OUTLINE:
{outline}

Instructions:
- Expand each bullet into clear, well-written paragraphs
- Maintain smooth flow and logical transitions
- Use simple, human-friendly language (avoid robotic tone)
- Add examples, scenarios, or mini case studies where helpful
- Add relevant keywords naturally (avoid keyword stuffing)
- Use storytelling elements when suitable
- Maintain high readability (short sentences & clean structure)
- Produce a blog post of 1200–1800 words
- End with a strong conclusion + key takeaway
- Include an FAQ section at the end (5–7 FAQs)

Return ONLY the final blog post content, with proper headings and formatting.
"""

    state["blog_post"]  = model.invoke(prompt).content
    return state

In [14]:
# Nodes
# Define your graph

graph = StateGraph(LLMState)

# nodes

graph.add_node('generate_outline',generate_outline)
graph.add_node('generate_detailed_blog',generate_detailed_blog)

# edges

graph.add_edge(START,'generate_outline')
graph.add_edge('generate_outline',"generate_detailed_blog")
graph.add_edge("generate_detailed_blog",END)


# compile

workflow = graph.compile()


In [15]:
# Execute the graph
initial_state = {'topic':'How to Set & Achieve Your Savings Goal (Backed by Psychology)'}
print(workflow.invoke(initial_state))

{'topic': 'How to Set & Achieve Your Savings Goal (Backed by Psychology)', 'outline': '**SEO‑Friendly Title**  \n*“Save Smarter, Not Harder: How to Set & Achieve Your Savings Goal (Backed by Psychology)”*\n\n---\n\n### Introduction (2–3 bullet points)\n- **Why mindset matters** – Briefly explain how psychological habits influence savings success.  \n- **The power of a clear target** – Highlight that specific, measurable goals outperform vague intentions.  \n- **What you’ll learn** – Outline the step‑by‑step, research‑backed framework readers will follow.\n\n---\n\n## H2: 1️⃣ Define Your Savings Vision  \n### H3: 1.1 Clarify the *Why*  \n- Talk points: personal motivation, life milestones, risk tolerance.  \n- Example: “Buying a house, emergency fund, or a dream vacation.”  \n\n### H3: 1.2 Set SMART, Sunk‑Cost‑Free Goals  \n- Talk points: Specific, Measurable, Achievable, Relevant, Time‑bound.  \n- Case‑study suggestion: A single mother saving for a child’s college fund over 5 years.\n\